# Stage 3 — Mars Network Explorer

| Field | Value |
|---|---|
| **Pipeline stage** | Stage 3 — Mars interactive network exploration |
| **Previous stage** | Stage 2 — Earth network exploration (`notebooks/analysis/06_earth_network_explorer.ipynb`) |
| **Next stage** | Stage 4 — Earth-Mars regime calibration (`notebooks/regime/00_calibration_overview.ipynb`) |
| **Purpose** | Browse Martian valley networks on a MOLA hillshade background. Inspect network structure, Strahler order distribution, and drainage density metrics. Provides the visual context for Earth-Mars calibration. |
| **Inputs** | `data/final_valleys/final_valleys.shp` (RAW_KEEP); `data/Mars/MOLA_Hillshade_Robinson_128ppd.tif`; `data/Mars/topology/mars_vn_topology_model_ready.gpkg` |
| **Outputs** | Network maps, summary tables. Nothing written to disk. |
| **Decision gate** | Informational. Use to identify source-data issues before Stage 4 calibration. |

## 0. Configuration

In [ ]:
# Number of example networks to show in the detail panel.
N_NETWORKS_DETAIL = 6

# Show MOLA hillshade as background (requires rasterio).
SHOW_HILLSHADE = True

# Set a specific network ID to highlight (None = show top N by node count).
HIGHLIGHT_NETWORK_ID = None

## 1. Imports and paths

In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

from channel_heads.io.paths import MARS_DIR, MARS_HILLSHADE, FINAL_VALLEYS_DIR
from channel_heads.dd_calibration import mars_network_table, mars_network_strahler

TOPOLOGY_GPKG = MARS_DIR / "topology" / "mars_vn_topology_model_ready.gpkg"
VALLEY_SHP    = FINAL_VALLEYS_DIR / "final_valleys.shp"

print(f"MOLA hillshade : {MARS_HILLSHADE}  exists={MARS_HILLSHADE.exists()}")
print(f"Topology GeoPackage : {TOPOLOGY_GPKG}  exists={TOPOLOGY_GPKG.exists()}")
print(f"Valley shapefile    : {VALLEY_SHP}  exists={VALLEY_SHP.exists()}")

## 2. Load Mars network summary table

In [ ]:
net_table = mars_network_table(TOPOLOGY_GPKG if TOPOLOGY_GPKG.exists() else None)

print(f"Networks loaded: {len(net_table)}")
print()
display(net_table.describe())

In [ ]:
# Sort by drainage density and show top 20
if 'drainage_density_km_km2' in net_table.columns:
    top = net_table.sort_values('drainage_density_km_km2', ascending=False).head(20)
    display(top)
elif 'n_nodes' in net_table.columns:
    top = net_table.sort_values('n_nodes', ascending=False).head(20)
    display(top)
else:
    display(net_table.head(20))

## 3. Drainage density distribution

In [ ]:
dd_col = next((c for c in net_table.columns if 'density' in c.lower()), None)

if dd_col:
    fig, ax = plt.subplots(figsize=(8, 4))
    dd_vals = net_table[dd_col].dropna()
    ax.hist(dd_vals, bins=50, color='steelblue', edgecolor='white')
    ax.axvline(dd_vals.median(), color='red', linestyle='--', label=f'Median {dd_vals.median():.2f}')
    ax.set_xlabel(dd_col)
    ax.set_ylabel('Network count')
    ax.set_title('Mars valley-network drainage density distribution')
    ax.legend()
    fig.tight_layout()
    plt.show()
    print(f"Median DD : {dd_vals.median():.3f} km/km²")
    print(f"Mean DD   : {dd_vals.mean():.3f} km/km²")
else:
    print("No drainage density column found. Available columns:", net_table.columns.tolist())

## 4. Strahler order distribution across all networks

In [ ]:
strahler_df = mars_network_strahler(TOPOLOGY_GPKG if TOPOLOGY_GPKG.exists() else None)

if strahler_df is not None and not strahler_df.empty:
    print("Strahler summary (rows = networks, cols = orders):")
    display(strahler_df.head(10))

    # Aggregate across all networks
    order_cols = [c for c in strahler_df.columns if str(c).isdigit()]
    if order_cols:
        totals = strahler_df[order_cols].sum()
        fig, ax = plt.subplots(figsize=(7, 3.5))
        ax.bar([int(c) for c in order_cols], totals.values, color='steelblue', edgecolor='white')
        ax.set_xlabel('Strahler order')
        ax.set_ylabel('Total node count (all networks)')
        ax.set_title('Mars valley-network Strahler order distribution')
        ax.set_xticks([int(c) for c in order_cols])
        fig.tight_layout()
        plt.show()
else:
    print("Strahler data not available or topology GeoPackage not found.")

## 5. Network overview map on MOLA hillshade

In [ ]:
try:
    import geopandas as gpd
    import rasterio
    from rasterio.enums import Resampling
    HAS_GEO = True
except ImportError as e:
    HAS_GEO = False
    print(f"Geographic packages not available ({e}). Skipping map.")

if HAS_GEO and VALLEY_SHP.exists() and MARS_HILLSHADE.exists() and SHOW_HILLSHADE:
    valleys = gpd.read_file(VALLEY_SHP)
    print(f"Valley features: {len(valleys)}  CRS: {valleys.crs}")

    with rasterio.open(MARS_HILLSHADE) as src:
        scale = 1024 / max(src.width, src.height)
        out_w = max(1, int(src.width  * scale))
        out_h = max(1, int(src.height * scale))
        hillshade = src.read(
            1, out_shape=(out_h, out_w),
            resampling=Resampling.bilinear
        ).astype(float)
        extent = [src.bounds.left, src.bounds.right,
                  src.bounds.bottom, src.bounds.top]
        hillshade_crs = src.crs

    valleys_reproj = valleys.to_crs(hillshade_crs) if valleys.crs != hillshade_crs else valleys

    fig, ax = plt.subplots(figsize=(14, 7))
    ax.imshow(hillshade, cmap='gray', extent=extent, aspect='auto',
              vmin=np.nanpercentile(hillshade, 5),
              vmax=np.nanpercentile(hillshade, 95))
    valleys_reproj.plot(ax=ax, color='cyan', linewidth=0.4, alpha=0.7)
    ax.set_title('Mars valley networks on MOLA hillshade', fontsize=12)
    ax.set_xlabel('Longitude (proj.)')
    ax.set_ylabel('Latitude (proj.)')
    fig.tight_layout()
    plt.show()

elif HAS_GEO and VALLEY_SHP.exists():
    valleys = gpd.read_file(VALLEY_SHP)
    fig, ax = plt.subplots(figsize=(14, 7))
    valleys.plot(ax=ax, color='steelblue', linewidth=0.5)
    ax.set_title('Mars valley networks (no hillshade — MOLA not found or SHOW_HILLSHADE=False)')
    fig.tight_layout()
    plt.show()

## 6. Per-network detail — top N networks

Show individual network maps for the largest (by node count or DD) networks.

In [ ]:
if HAS_GEO and TOPOLOGY_GPKG.exists():
    import warnings
    nodes_gdf = None
    edges_gdf = None
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        try:
            edges_gdf = gpd.read_file(TOPOLOGY_GPKG, layer='edges')
        except Exception:
            pass
        try:
            nodes_gdf = gpd.read_file(TOPOLOGY_GPKG, layer='nodes')
        except Exception:
            pass

    if edges_gdf is not None and 'network_id' in edges_gdf.columns:
        net_id_col = 'network_id'
        top_nets = (
            edges_gdf.groupby(net_id_col).size()
            .sort_values(ascending=False)
            .head(N_NETWORKS_DETAIL)
            .index.tolist()
        )
        if HIGHLIGHT_NETWORK_ID is not None:
            top_nets = [HIGHLIGHT_NETWORK_ID]

        ncols = min(3, len(top_nets))
        nrows = -(-len(top_nets) // ncols)
        fig, axes = plt.subplots(nrows, ncols,
                                 figsize=(5 * ncols, 4.5 * nrows))
        axes = np.array(axes).flatten()

        for i, nid in enumerate(top_nets):
            ax = axes[i]
            sub = edges_gdf[edges_gdf[net_id_col] == nid]
            sub.plot(ax=ax, color='steelblue', linewidth=1.2)
            if nodes_gdf is not None and net_id_col in nodes_gdf.columns:
                nsub = nodes_gdf[nodes_gdf[net_id_col] == nid]
                outlet = nsub[nsub.get('node_type', pd.Series()) == 'outlet'] if 'node_type' in nsub.columns else None
                if outlet is not None and not outlet.empty:
                    outlet.plot(ax=ax, color='red', markersize=6, zorder=5)
            ax.set_title(f'Network {nid}\n({len(sub)} edges)', fontsize=9)
            ax.axis('off')

        for j in range(len(top_nets), len(axes)):
            axes[j].axis('off')

        fig.suptitle(f'Top {len(top_nets)} Mars networks by edge count', fontsize=11)
        fig.tight_layout()
        plt.show()
    else:
        print("Edges layer not found or no 'network_id' column. Check topology GeoPackage.")
else:
    print("geopandas not available or topology GeoPackage not found. Skipping detail view.")